# xbrl-frontier: student predictions with vLLM on a Kaggle T4 (owner-run)

Writes one predictions file per target (zero-shot base and/or LoRA adapters) for `SET`. Scoring happens on the Mac (`scripts/score_predictions.py`).
**Settings:** GPU T4, Internet on. **Attach:** `xbrl-frontier-src`, `xbrl-frontier-corpus` (for `dev`), and the run output datasets holding the adapters.

In [ ]:
MODEL = 'Qwen/Qwen3-1.7B'
SET = 'dev'                       # 'dev', 'finni' or 'fresh'
BASE = False                      # also predict zero-shot with the base model
ADAPTERS = {}                     # name -> adapter dir, e.g. {'1p7b-s1-prefix_2500': '/kaggle/input/xbrl-frontier-run-1p7b-s1/curve/prefix_2500'}
CHECKPOINT = None                 # 'NAME=DIR' to serve an exported checkpoint (merged fp16 / INT8 / AWQ) instead of BASE/ADAPTERS
RESUME = None                     # previous session's output dir (attach that kernel via --kernel-source) to continue a stopped run


In [ ]:
import glob, os, shutil, subprocess, sys
dst = '/kaggle/working/xbrl-frontier'
tarballs = glob.glob('/kaggle/input/**/xbrl-frontier-src.tar.gz', recursive=True)
trees = [os.path.dirname(os.path.dirname(os.path.dirname(p)))
         for p in glob.glob('/kaggle/input/**/docs/config/thresholds.yaml', recursive=True)]
if tarballs:
    assert len(tarballs) == 1, tarballs
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['tar', '-xzf', tarballs[0], '-C', dst], check=True)
else:  # Kaggle auto-extracted the upload
    assert len(trees) == 1, trees
    shutil.copytree(trees[0], dst, dirs_exist_ok=True)
os.chdir(dst)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import yaml
pins = yaml.safe_load(open('docs/config/thresholds.yaml'))['pins']['software']
missing = [k for k in ('vllm',) if not pins.get(k)]
assert not missing, f'pins.software empty for {missing}: owner approval needed first'
print(pins)

In [ ]:
!pip install -q "vllm=={pins['vllm']}" "pyyaml==6.0.3" "pyarrow==25.0.1"
!pip uninstall -y -q torchaudio  # Kaggle's torchaudio is built for another CUDA than vllm's torch; transformers imports it
import importlib.metadata as md
keys = ('vllm',)
installed = {k: md.version(k) for k in keys}
bad = {k: (v, pins[k]) for k, v in installed.items() if v != pins[k]}
assert not bad, f'installed != pins.software: {bad}; restart the kernel and run all again'
import importlib
imported = {k: importlib.import_module(k).__version__ for k in keys}
bad = {k: (v, pins[k]) for k, v in imported.items() if v != pins[k]}
assert not bad, f'imported != pins.software: {bad}; restart the kernel (Run > Restart) and run all again'
import torch; print(imported, 'torch', torch.__version__, 'CUDA', torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
import glob, os
hits = glob.glob('/kaggle/input/**/dev.jsonl', recursive=True)
corpus = os.path.dirname(hits[0]) if hits else None
out = f'/kaggle/working/predictions-{MODEL.split("/")[-1]}-{SET}'
cmd = f'python -m serving.generate_predictions --model {MODEL} --set {SET} --out {out}'
cmd += f' --corpus {corpus}' if corpus else ''
cmd += ' --base' if BASE else ''
cmd += ''.join(f' --adapter {k}={v}' for k, v in ADAPTERS.items())
cmd += f' --checkpoint {CHECKPOINT}' if CHECKPOINT else ''
cmd += f' --resume {RESUME}' if RESUME else ''
print(cmd)
!{cmd}
!ls -la {out}
import json as _json
_meta = _json.load(open(f'{out}/predictions_{SET}.meta.json'))
print('complete', _meta['complete'], {k: (v['written'], v['complete']) for k, v in _meta['targets'].items()})
print('configured max_model_len', _meta['configured_max_model_len'], '| effective (accepted by vLLM)', _meta['effective_max_model_len'],
      '| refusals', _meta['max_model_len_refusals'], '| not_generated', _meta['not_generated_over_max_model_len'])